In [ ]:
import os
from pathlib import Path
from functools import reduce
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [2]:
# get all csv filepaths as aboslute paths
data_csvs = list(map(Path.resolve, Path('../data/').rglob("*.csv")))
for fp in data_csvs:
    print(fp)

/data1/repos/DSC445-ML1-Election-Prediction/data/Raw/countypres_2000-2024.csv
/data1/repos/DSC445-ML1-Election-Prediction/data/Raw/demographic/ACSDT5Y2020.B01001-Data.csv
/data1/repos/DSC445-ML1-Election-Prediction/data/Raw/demographic/ACSDT5Y2020.B02001-Data.csv
/data1/repos/DSC445-ML1-Election-Prediction/data/Raw/demographic/ACSDT5Y2020.B15003-Data.csv
/data1/repos/DSC445-ML1-Election-Prediction/data/Raw/socioeconomic/ACSDT5Y2020.B25001-Data.csv
/data1/repos/DSC445-ML1-Election-Prediction/data/Raw/socioeconomic/ACSDT5Y2020.B17001-Data.csv
/data1/repos/DSC445-ML1-Election-Prediction/data/Raw/socioeconomic/ACSDT5Y2020.B19013-Data.csv
/data1/repos/DSC445-ML1-Election-Prediction/data/Raw/socioeconomic/ACSDT5Y2020.B23025-Data.csv


## Election Dataset

Column Summary

| Columns        | Description                                |
| -------------- | ------------------------------------------ |
| state          | Full state name                            |
| state_po       | State abbreviation                         |
| county_name    | County name                                |
| county_fips    | County FIPS code used for merging datasets |
| year           | Election year                              |
| office         | Election office (US PRESIDENT)             |
| candidate      | Candidate name                             |
| party          | Political party                            |
| candidatevotes | Votes received by candidate                |
| totalvotes     | Total votes in county                      |

Column Status:

- State/State_PO: duplicate data set
- county_name: good info, but provide duplicate data to county_fips, also includes state data
- county_fips: has missing data, but can be addressed easily with dummy data
- year: used for splitting dataset by year
- office: only contains 'president', not needed
- candidate name: not needed as the prediction is done by party but is a good reference
- candidate votes/total votes: self explanatory, good continuous values

Overall, 

This dataset will be **split** along `year`, provides **label** through `candidate` or `party`, and `county_fips`, `candidatevotes`, `totalvotes` as **features**. 




In [3]:
election_csv = data_csvs[0]
# import as strings
election_df = pd.read_csv(election_csv, dtype={'county_fips': str})

## Basic Data Check

In [4]:
election_df.head()

,state,county_name,year,state_po,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,OTHER,OTHER,293.0,28281,20260225,TOTAL
1,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,CHASE OLIVER,LIBERTARIAN,65.0,28281,20260225,TOTAL
2,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,KAMALA D HARRIS,DEMOCRAT,7439.0,28281,20260225,TOTAL
3,ALABAMA,AUTAUGA,2024,AL,1001,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,20484.0,28281,20260225,TOTAL
4,ALABAMA,BALDWIN,2024,AL,1003,US PRESIDENT,OTHER,OTHER,1276.0,122249,20260225,TOTAL


In [5]:
election_df.office.unique()

<ArrowStringArray>
['US PRESIDENT']
Length: 1, dtype: str

### Review nan Values

In [6]:
nan_columns = []
for col in election_df.columns:
    nan_count = election_df[col].isna().sum()
    if nan_count:
        print(f'{col} np.nan count: {nan_count}')
        nan_columns.append(col)

county_fips np.nan count: 52
party np.nan count: 501
candidatevotes np.nan count: 37
mode np.nan count: 2795


In [7]:
nan_rows_per_columns = {
    n_col: election_df.index[election_df[n_col].isna()]
    for n_col in nan_columns
}

#### Missing County Fips and Party

Missing County Fips for these locations can be dropped, since they are not part of the other dataset

Missing party can also be dropped since it is not part of the 2 parties for prediction

In [8]:
def get_missing_sets(indices: pd.Index,
                     set_columns: list,
                     df: pd.DataFrame=election_df) -> set:
    """helper for getting a set of values to check for missing values"""
    _df = df.loc[indices, set_columns]
    return _df.drop_duplicates().values

In [9]:
missing_fips = get_missing_sets(nan_rows_per_columns['county_fips'],
                                ('state', 'county_name'))
print(missing_fips)

[['CONNECTICUT' 'STATEWIDE WRITEIN']
 ['MAINE' 'MAINE UOCAVA']
 ['RHODE ISLAND' 'FEDERAL PRECINCT']]


In [10]:
missing_party = get_missing_sets(nan_rows_per_columns['candidatevotes'],
                                ('state', 'candidate', 'party'))
print(missing_party)

[['NEW MEXICO' 'CHASE OLIVER' 'LIBERTARIAN']]


### Parsing and Updating Voting Data

In [287]:
# parse and copy
parsed_election_df = election_df.loc[election_df.index[election_df['mode']=='TOTAL']]
parsed_election_df = parsed_election_df.loc[:, ['county_fips', 'party', 'year', 'candidatevotes', 'totalvotes']].copy()
# drop na values
parsed_election_df = parsed_election_df.dropna()
biparty_idx = parsed_election_df.index[parsed_election_df.party.isin(['DEMOCRAT', 'REPUBLICAN'])]
parsed_election_df = parsed_election_df.loc[biparty_idx]
parsed_election_df['county_fips'] = parsed_election_df.county_fips.astype(int)

In [288]:
dem_votes = parsed_election_df.party == 'DEMOCRAT'
dem_index = parsed_election_df.index[dem_votes]
rep_index = parsed_election_df.index[~dem_votes]
# copy for further processing
dem_election_df = parsed_election_df.loc[dem_index].copy()
rep_election_df = parsed_election_df.loc[rep_index].copy()
vote_df = pd.merge(dem_election_df, rep_election_df, on=['county_fips', 'year'], how='inner', suffixes=('_dem', '_rep'))

In [289]:
# add winner column
vote_df['winner'] = vote_df.candidatevotes_dem > vote_df.candidatevotes_rep
vote_df['winner'] = vote_df['winner'].map(lambda x: 'DEMOCRAT' if x else 'REPUBLICAN')
# drop redundant party info
vote_df = vote_df.drop(columns=['party_dem', 'party_rep', 'totalvotes_rep'])
vote_df = vote_df.rename(columns={'totalvotes_dem':'totalvotes'})

## Socioeconomic and Demographic Dataset

**Note**

1. ACSDT5Y2020.B25001-Data.csv does not provide any additional data and should be skipped
2. ACSDT5Y2020.B19013-Data.csv provides income data and should be normalized differently



Other dataset requires some cleaning and modification

1. The first row provides column descriptions, which serve better as column names. The column names also should be renamed
2. The last column is imported is `nan` due to them having trailing errors.
3. Margin of error columns are not needed as they are not relevant for predictive purposes
4. Use the provided total to normalize dataset


In [114]:
def load_csv(filepath: Path) -> pd.DataFrame:
    """helper for loading dataset"""
    # drop first row 
    df = pd.read_csv(filepath, header=1).iloc[:, :-1]
    # clean and filter columns
    cols = df.columns
    col_idxs = list(range(0, len(cols), 2))
    clean_cols = []
    # skip every other column, last column is not needed
    for i in col_idxs:
        col = cols[i]
        # replace unnecessary characters
        col = col.replace('Estimate!!', '')
        col = col.replace(':!!', ' ')
        col = col.replace(':', '')
        clean_cols.append(col)
    df = df.iloc[:, col_idxs]
    df.columns = clean_cols
    return df

### Socioeconimic Dataset Processing

In [200]:
socio_dfs = []
for fp in data_csvs[1:4]:
    _df = load_csv(fp)
    # normalize columns
    for col in _df.columns[2:]:
        _df[col] = _df.loc[:, col] / _df.loc[:, 'Total']
    # drop total
    _df = _df.drop(columns=['Total'])
    socio_dfs.append(_df)
# merge
socio_df = reduce(lambda left, right: pd.merge(left, right, on='Geography', how='outer'),
                  socio_dfs)

In [201]:
socio_df.head()

,Geography,Total Male,Total Male Under 5 years,Total Male 5 to 9 years,Total Male 10 to 14 years,Total Male 15 to 17 years,Total Male 18 and 19 years,Total Male 20 years,Total Male 21 years,Total Male 22 to 24 years,...,"Total 12th grade, no diploma",Total Regular high school diploma,Total GED or alternative credential,"Total Some college, less than 1 year","Total Some college, 1 or more years, no degree",Total Associate's degree,Total Bachelor's degree,Total Master's degree,Total Professional school degree,Total Doctorate degree
0,0500000US01001,0.486206,0.031039,0.038750,0.030087,0.021711,0.009939,0.007099,0.004044,0.020256,...,0.017644,0.255230,0.058558,0.057950,0.144453,0.087771,0.166931,0.089778,0.013312,0.013154
1,0500000US01003,0.485086,0.027862,0.023808,0.037973,0.019722,0.011022,0.005392,0.003628,0.016405,...,0.021875,0.221910,0.049825,0.067567,0.154047,0.092291,0.202130,0.085695,0.018288,0.012959
2,0500000US01005,0.525693,0.026373,0.029609,0.028810,0.018661,0.010429,0.014585,0.004116,0.017262,...,0.027477,0.290555,0.066865,0.057201,0.138731,0.077822,0.072821,0.030342,0.006630,0.006349
3,0500000US01007,0.537320,0.027979,0.030973,0.027755,0.025431,0.014436,0.004961,0.003173,0.023420,...,0.023019,0.367736,0.083005,0.048039,0.128604,0.067993,0.073998,0.030087,0.006818,0.002565
4,0500000US01009,0.496528,0.031928,0.030093,0.035218,0.021764,0.012865,0.006545,0.005402,0.017713,...,0.019566,0.265786,0.085221,0.085774,0.127844,0.131110,0.088913,0.034711,0.005576,0.003315


### Demographics Dataset

In [202]:
demo_dfs = []
for fp in data_csvs[5:]:
    _df = load_csv(fp)
    # different processing for income
    if fp.name == 'ACSDT5Y2020.B19013-Data.csv':
        # update values
        _df[_df.columns[1]] = _df.iloc[:, 1].map(lambda x: 0 if x == '-' else int(x))
        _df[_df.columns[1]] = _df.iloc[:, 1] / _df.iloc[:, 1].mean()
        demo_dfs.append(_df)
    else:
        for col in _df.columns[2:]:
            _df[col] = _df.loc[:, col] / _df.loc[:, 'Total']
            # drop total
        _df = _df.drop(columns=['Total'])
        demo_dfs.append(_df)
# merge
demo_df = reduce(lambda left, right: pd.merge(left, right, on='Geography', how='outer'),
                 demo_dfs)

In [203]:
demo_df.head()

,Geography,Total Income in the past 12 months below poverty level,Total Income in the past 12 months below poverty level Male,Total Income in the past 12 months below poverty level Male Under 5 years,Total Income in the past 12 months below poverty level Male 5 years,Total Income in the past 12 months below poverty level Male 6 to 11 years,Total Income in the past 12 months below poverty level Male 12 to 14 years,Total Income in the past 12 months below poverty level Male 15 years,Total Income in the past 12 months below poverty level Male 16 and 17 years,Total Income in the past 12 months below poverty level Male 18 to 24 years,...,Total Income in the past 12 months at or above poverty level Female 55 to 64 years,Total Income in the past 12 months at or above poverty level Female 65 to 74 years,Total Income in the past 12 months at or above poverty level Female 75 years and over,Median household income in the past 12 months (in 2020 inflation-adjusted dollars),Total In labor force,Total In labor force Civilian labor force,Total In labor force Civilian labor force Employed,Total In labor force Civilian labor force Unemployed,Total In labor force Armed Forces,Total Not in labor force
0,0500000US01001,0.152118,0.061923,0.004657,0.000997,0.006560,0.004422,0.001885,0.004096,0.006470,...,0.054457,0.044581,0.032783,1.070674,0.586677,0.570694,0.554103,0.016592,0.015983,0.413323
1,0500000US01003,0.091737,0.036264,0.002617,0.000818,0.002147,0.001952,0.000771,0.001241,0.004513,...,0.066533,0.060496,0.039085,1.140363,0.583064,0.581249,0.558478,0.022770,0.001815,0.416936
2,0500000US01005,0.285999,0.113684,0.019937,0.004259,0.012324,0.009017,0.006479,0.001676,0.019937,...,0.055324,0.057952,0.034708,0.646112,0.458470,0.458470,0.426667,0.031803,0.000000,0.541530
3,0500000US01007,0.180981,0.089888,0.009061,0.002072,0.011278,0.004145,0.006555,0.005350,0.010844,...,0.052776,0.046125,0.039522,0.955060,0.486944,0.486944,0.450736,0.036209,0.000000,0.513056
4,0500000US01009,0.137361,0.057381,0.004866,0.001260,0.007405,0.002048,0.000788,0.002398,0.003851,...,0.062020,0.046703,0.033504,0.903375,0.523686,0.522731,0.495541,0.027190,0.000955,0.476314


### Combine Socio-economic and Demography

Combine dataset and update Geography as county fips

In [271]:
norm_data = pd.merge(socio_df, demo_df, on='Geography', how='inner')
norm_data = norm_data.rename(columns={'Geography': 'county_fips'})
norm_data['county_fips'] = norm_data['county_fips'].map(lambda x: int(x.split('US')[-1]))

## Combine Election Dataset with Normalized Dataset

In [291]:
joined_df = pd.merge(vote_df, norm_data, on='county_fips', how='left')

### Review only 2020 Dataset

Since dataset only has 2020.

In [292]:
processed_2020_df = joined_df.loc[joined_df.index[joined_df.year==2020]].copy()
processed_2020_df.to_csv('../data/processed/2020_joined_norm_jeong.csv', index=False)

# EDA

In [16]:
eda_df = pd.read_csv('../data/processed/2020_joined_norm_jeong.csv')

In [28]:
eda_df.head()

,county_fips,year,candidatevotes_dem,totalvotes,candidatevotes_rep,winner,Total Male,Total Male Under 5 years,Total Male 5 to 9 years,Total Male 10 to 14 years,...,Total Income in the past 12 months at or above poverty level Female 55 to 64 years,Total Income in the past 12 months at or above poverty level Female 65 to 74 years,Total Income in the past 12 months at or above poverty level Female 75 years and over,Median household income in the past 12 months (in 2020 inflation-adjusted dollars),Total In labor force,Total In labor force Civilian labor force,Total In labor force Civilian labor force Employed,Total In labor force Civilian labor force Unemployed,Total In labor force Armed Forces,Total Not in labor force
0,1001,2020,7503.0,27770,19838.0,REPUBLICAN,0.486206,0.031039,0.038750,0.030087,...,0.054457,0.044581,0.032783,1.070674,0.586677,0.570694,0.554103,0.016592,0.015983,0.413323
1,1003,2020,24578.0,109679,83544.0,REPUBLICAN,0.485086,0.027862,0.023808,0.037973,...,0.066533,0.060496,0.039085,1.140363,0.583064,0.581249,0.558478,0.022770,0.001815,0.416936
2,1005,2020,4816.0,10518,5622.0,REPUBLICAN,0.525693,0.026373,0.029609,0.028810,...,0.055324,0.057952,0.034708,0.646112,0.458470,0.458470,0.426667,0.031803,0.000000,0.541530
3,1007,2020,1986.0,9595,7525.0,REPUBLICAN,0.537320,0.027979,0.030973,0.027755,...,0.052776,0.046125,0.039522,0.955060,0.486944,0.486944,0.450736,0.036209,0.000000,0.513056
4,1009,2020,2640.0,27588,24711.0,REPUBLICAN,0.496528,0.031928,0.030093,0.035218,...,0.062020,0.046703,0.033504,0.903375,0.523686,0.522731,0.495541,0.027190,0.000955,0.476314


In [31]:
# dataset that does not need
numeric_data = eda_df.iloc[:, 6:]
# encodeing fips
encoder = OneHotEncoder(sparse_output=False)
encoded_data = encoder.fit_transform(eda_df.iloc[:, 0].values.reshape(-1, 1))
fips_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(['fips']))
dataset_df = pd.concat([fips_df, numeric_data], axis=1)
dataset_columns = dataset_df.columns

In [67]:
numeric_data.shape

(2305, 146)

In [34]:
# labels
# classification label
class_label = eda_df.winner.map({'REPUBLICAN': 0, 'DEMOCRAT':1})
dem_reg_label = eda_df.candidatevotes_dem / eda_df.totalvotes
rep_reg_label = eda_df.candidatevotes_rep / eda_df.totalvotes
# combine
label_df = pd.DataFrame([class_label, dem_reg_label, rep_reg_label]).T
label_df.columns = ['class_label', 'dem_reg_votes', 'rep_reg_votes']

In [35]:
# continuous data only for correlation since FIPS are one hot
cont_data_df = pd.concat([label_df, numeric_data], axis=1)


### Correlation

In [63]:
# correlation
corr_mat = cont_data_df.corr()
dem_winner_top_features = corr_mat['class_label'].sort_values(ascending=False)
dem_vote_top_features = corr_mat['dem_reg_votes'].sort_values(ascending=False)
rep_vote_top_features = corr_mat['rep_reg_votes'].sort_values(ascending=False)

In [72]:
for lbl, data in zip(['class label', 'dem votes', 'rep votes'],
                     [dem_winner_top_features, dem_vote_top_features, rep_vote_top_features]):
    print(f'{lbl} Top Features')
    print(data.iloc[2:17])
    print('\n')
    print(f'{lbl} Bottom Features')
    print(data.iloc[-17:])
    print('\n')

class label Top Features
Total Master's degree                                                                 0.499365
Total Professional school degree                                                      0.478159
Total Asian alone                                                                     0.416043
Total Doctorate degree                                                                0.411543
Total Bachelor's degree                                                               0.392745
Total Female 25 to 29 years                                                           0.292393
Total Black or African American alone                                                 0.288043
Total Income in the past 12 months at or above poverty level Female 25 to 34 years    0.276368
Total In labor force Civilian labor force Unemployed                                  0.271090
Total Female 30 to 34 years                                                           0.262302
Total Two or more races  

### Deviations

In [71]:
StdDevs = numeric_data.std().sort_values(ascending=False)

print('Most Deviation')
print(StdDevs.iloc[:15])

print('Least Deviation')
print(StdDevs.iloc[-15:])

Most Deviation
Median household income in the past 12 months (in 2020 inflation-adjusted dollars)    0.259199
Total White alone                                                                     0.153814
Total Black or African American alone                                                 0.124833
Total In labor force Civilian labor force Employed                                    0.080279
Total Not in labor force                                                              0.077691
Total In labor force                                                                  0.077691
Total In labor force Civilian labor force                                             0.077297
Total American Indian and Alaska Native alone                                         0.071855
Total Regular high school diploma                                                     0.066661
Total Income in the past 12 months below poverty level                                0.059956
Total Income in the past 12 months 

# OLD

In [175]:
demo_dfs = []
for fp in data_csvs[5:]:
    _df = load_csv(fp)
    demo_dfs.append(_df)

In [167]:
len(demo_dfs)

3

In [177]:
demo_dfs[1].iloc[:, 1].map(lambda x: 0 if x == '-' else int(x), in_place=True)

TypeError: <lambda>() got an unexpected keyword argument 'in_place'

In [157]:
'Total' in demo_dfs[0].columns

True

In [135]:
demo_dfs[1]

,Geography,Median household income in the past 12 months (in 2020 inflation-adjusted dollars)
0,0500000US01001,57982
1,0500000US01003,61756
2,0500000US01005,34990
3,0500000US01007,51721
4,0500000US01009,48922
...,...,...
3216,0500000US72145,20126
3217,0500000US72147,14040
3218,0500000US72149,19355
3219,0500000US72151,16828


In [158]:
'Total' in demo_dfs[2].columns

True

In [ ]:
def load_csv(filepath: Path, drop_total: bool=True) -> pd.DataFrame:
    """helper for loading dataset"""
    # drop first row 
    df = pd.read_csv(filepath, header=1).iloc[:, :-1]
    # clean and filter columns
    cols = df.columns
    col_idxs = list(range(0, len(cols), 2))
    clean_cols = []
    # skip every other column, last column is not needed
    for i in col_idxs:
        col = cols[i]
        # replace unnecessary characters
        col = col.replace('Estimate!!', '')
        col = col.replace(':!!', ' ')
        col = col.replace(':', '')
        clean_cols.append(col)
    df = df.iloc[:, col_idxs]
    df.columns = clean_cols
    # drop total if found
    if drop_total and 'Total' in df.columns:
        df = df.drop(columns='Total')
    return df

def load_combined(filepaths: list) -> pd.DataFrame:
    dfs = map(load_csv, filepaths)
    df = reduce(lambda left, right: pd.merge(left,
                                             right,
                                             on='Geography',
                                             how='outer'),
                dfs)
    return df

In [ ]:
# load dataset
total_df = load_csv(data_csvs[4], False)
socio_df = load_combined(data_csvs[1:4])
demo_df = load_combined(data_csvs[5:])

In [112]:
_socio = pd.merge(total_df, socio_df, on='Geography', how='inner')

In [113]:
_socio

,Geography,Total,Total Male,Total Male Under 5 years,Total Male 5 to 9 years,Total Male 10 to 14 years,Total Male 15 to 17 years,Total Male 18 and 19 years,Total Male 20 years,Total Male 21 years,...,"Total 12th grade, no diploma",Total Regular high school diploma,Total GED or alternative credential,"Total Some college, less than 1 year","Total Some college, 1 or more years, no degree",Total Associate's degree,Total Bachelor's degree,Total Master's degree,Total Professional school degree,Total Doctorate degree
0,0500000US01001,23697,27052,1727,2156,1674,1208,553,395,225,...,668,9663,2217,2194,5469,3323,6320,3399,504,498
1,0500000US01003,116747,105889,6082,5197,8289,4305,2406,1177,792,...,3403,34521,7751,10511,23964,14357,31444,13331,2845,2016
2,0500000US01005,12057,13156,660,741,721,467,261,365,103,...,489,5171,1190,1018,2469,1385,1296,540,118,113
3,0500000US01007,9237,12022,626,693,621,569,323,111,71,...,368,5879,1327,768,2056,1087,1183,481,109,41
4,0500000US01009,24404,28677,1844,1738,2034,1257,743,378,312,...,779,10582,3393,3415,5090,5220,3540,1382,222,132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3216,0500000US72145,24677,24420,1081,1250,1635,1005,685,474,156,...,851,9644,815,581,3802,4167,5707,1776,182,283
3217,0500000US72147,4940,4356,316,195,223,159,50,135,0,...,146,2523,203,0,169,566,434,141,0,207
3218,0500000US72149,9303,10589,551,644,658,497,370,139,183,...,36,5275,349,328,1362,1520,2413,474,51,32
3219,0500000US72151,14648,15883,619,815,1064,640,442,208,368,...,170,5634,506,224,3657,2847,3567,490,44,85


In [110]:
socio_df

,Geography,Total Male,Total Male Under 5 years,Total Male 5 to 9 years,Total Male 10 to 14 years,Total Male 15 to 17 years,Total Male 18 and 19 years,Total Male 20 years,Total Male 21 years,Total Male 22 to 24 years,...,"Total 12th grade, no diploma",Total Regular high school diploma,Total GED or alternative credential,"Total Some college, less than 1 year","Total Some college, 1 or more years, no degree",Total Associate's degree,Total Bachelor's degree,Total Master's degree,Total Professional school degree,Total Doctorate degree
0,0500000US01001,27052,1727,2156,1674,1208,553,395,225,1127,...,668,9663,2217,2194,5469,3323,6320,3399,504,498
1,0500000US01003,105889,6082,5197,8289,4305,2406,1177,792,3581,...,3403,34521,7751,10511,23964,14357,31444,13331,2845,2016
2,0500000US01005,13156,660,741,721,467,261,365,103,432,...,489,5171,1190,1018,2469,1385,1296,540,118,113
3,0500000US01007,12022,626,693,621,569,323,111,71,524,...,368,5879,1327,768,2056,1087,1183,481,109,41
4,0500000US01009,28677,1844,1738,2034,1257,743,378,312,1023,...,779,10582,3393,3415,5090,5220,3540,1382,222,132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3216,0500000US72145,24420,1081,1250,1635,1005,685,474,156,1109,...,851,9644,815,581,3802,4167,5707,1776,182,283
3217,0500000US72147,4356,316,195,223,159,50,135,0,271,...,146,2523,203,0,169,566,434,141,0,207
3218,0500000US72149,10589,551,644,658,497,370,139,183,490,...,36,5275,349,328,1362,1520,2413,474,51,32
3219,0500000US72151,15883,619,815,1064,640,442,208,368,499,...,170,5634,506,224,3657,2847,3567,490,44,85


In [111]:
demo_df

,Geography,Total Income in the past 12 months below poverty level,Total Income in the past 12 months below poverty level Male,Total Income in the past 12 months below poverty level Male Under 5 years,Total Income in the past 12 months below poverty level Male 5 years,Total Income in the past 12 months below poverty level Male 6 to 11 years,Total Income in the past 12 months below poverty level Male 12 to 14 years,Total Income in the past 12 months below poverty level Male 15 years,Total Income in the past 12 months below poverty level Male 16 and 17 years,Total Income in the past 12 months below poverty level Male 18 to 24 years,...,Total Income in the past 12 months at or above poverty level Female 55 to 64 years,Total Income in the past 12 months at or above poverty level Female 65 to 74 years,Total Income in the past 12 months at or above poverty level Female 75 years and over,Median household income in the past 12 months (in 2020 inflation-adjusted dollars),Total In labor force,Total In labor force Civilian labor force,Total In labor force Civilian labor force Employed,Total In labor force Civilian labor force Unemployed,Total In labor force Armed Forces,Total Not in labor force
0,0500000US01001,8394,3417,257,55,362,244,104,226,357,...,3005,2460,1809,57982,26025,25316,24580,736,709,18335
1,0500000US01003,19739,7803,563,176,462,420,166,267,971,...,14316,13017,8410,61756,103116,102795,98768,4027,321,73736
2,0500000US01005,6312,2509,440,94,272,199,143,37,440,...,1221,1279,766,34990,9356,9356,8707,649,0,11051
3,0500000US01007,3755,1865,188,43,234,86,136,111,225,...,1095,957,820,51721,8970,8970,8303,667,0,9451
4,0500000US01009,7847,3278,278,72,423,117,45,137,220,...,3543,2668,1914,48922,24133,24089,22836,1253,44,21950
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3216,0500000US72145,23229,10633,742,66,943,478,80,406,1090,...,2161,1617,1393,20126,18341,18322,15013,3309,19,24396
3217,0500000US72147,4438,2151,246,54,107,189,45,78,142,...,375,339,195,14040,2363,2363,2053,310,0,4720
3218,0500000US72149,10100,4638,411,72,443,216,37,154,549,...,859,843,385,19355,8139,8137,6506,1631,2,9890
3219,0500000US72151,16966,7824,450,80,751,310,94,273,985,...,1494,757,870,16828,10556,10556,8644,1912,0,17105


In [104]:
total_df.head()

,Geography,Total
0,0500000US01001,23697
1,0500000US01003,116747
2,0500000US01005,12057
3,0500000US01007,9237
4,0500000US01009,24404


In [95]:
cols = {}
for fp in data_csvs[1:]:
    cols[fp.name] = load_csv(fp).columns

In [99]:
data_csvs[4]

PosixPath('/data1/repos/DSC445-ML1-Election-Prediction/data/Raw/socioeconomic/ACSDT5Y2020.B25001-Data.csv')

In [ ]:
pd.read_csv(b)

,GEO_ID,NAME,B25001_001E,B25001_001M,Unnamed: 4
0,Geography,Geographic Area Name,Estimate!!Total,Margin of Error!!Total,NaN
1,0500000US01001,"Autauga County, Alabama",23697,68,NaN
2,0500000US01003,"Baldwin County, Alabama",116747,180,NaN
3,0500000US01005,"Barbour County, Alabama",12057,119,NaN
4,0500000US01007,"Bibb County, Alabama",9237,82,NaN
...,...,...,...,...,...
3217,0500000US72145,"Vega Baja Municipio, Puerto Rico",24677,271,NaN
3218,0500000US72147,"Vieques Municipio, Puerto Rico",4940,221,NaN
3219,0500000US72149,"Villalba Municipio, Puerto Rico",9303,178,NaN
3220,0500000US72151,"Yabucoa Municipio, Puerto Rico",14648,277,NaN


In [96]:
cols

{'ACSDT5Y2020.B01001-Data.csv': Index(['Geography', 'Estimate!!Total:', 'Estimate!!Total:!!Male:',
        'Estimate!!Total:!!Male:!!Under 5 years',
        'Estimate!!Total:!!Male:!!5 to 9 years',
        'Estimate!!Total:!!Male:!!10 to 14 years',
        'Estimate!!Total:!!Male:!!15 to 17 years',
        'Estimate!!Total:!!Male:!!18 and 19 years',
        'Estimate!!Total:!!Male:!!20 years',
        'Estimate!!Total:!!Male:!!21 years',
        'Estimate!!Total:!!Male:!!22 to 24 years',
        'Estimate!!Total:!!Male:!!25 to 29 years',
        'Estimate!!Total:!!Male:!!30 to 34 years',
        'Estimate!!Total:!!Male:!!35 to 39 years',
        'Estimate!!Total:!!Male:!!40 to 44 years',
        'Estimate!!Total:!!Male:!!45 to 49 years',
        'Estimate!!Total:!!Male:!!50 to 54 years',
        'Estimate!!Total:!!Male:!!55 to 59 years',
        'Estimate!!Total:!!Male:!!60 and 61 years',
        'Estimate!!Total:!!Male:!!62 to 64 years',
        'Estimate!!Total:!!Male:!!65 and 66 yea

## Socioeconomic Dataset

This dataset requires some cleaning. 

1. The first row provides column descriptions, which serve better as column names. The column names also should be renamed
2. The last column is imported is `nan` due to them having trailing errors.
3. Margin of error columns are not needed as they are not relevant for predictive purposes

In [84]:
def load_eco_demo_csv(filepath: Path) -> pd.DataFrame:
    """helper for loading economic dataset"""
    # drop first row 
    df = pd.read_csv(filepath, header=1).iloc[:, :-1]
    # clean and filter columns
    cols = df.columns
    col_idxs = list(range(0, len(cols), 2))
    clean_cols = []
    # skip every other column, last column is not needed
    for i in col_idxs:
        col = cols[i]
        # replace unnecessary characters
        col = col.replace('Estimate!!', '')
        col = col.replace(':!!', ' ')
        col = col.replace(':', '')
        clean_cols.append(col)
    df = df.iloc[:, col_idxs]
    df.columns = clean_cols
    # drop total if found
    if 'Total' in df.columns:
        df = df.drop(columns='Total')
    return df

from functools import reduce

def load_combined_eco_demo(filepaths: list) -> pd.DataFrame:
    dfs = [load_eco_demo_csv(fp) for fp in filepaths]
    df = reduce(lambda left, right: pd.merge(left,
                                             right,
                                             on='Geography',
                                             how='outer'),
                dfs)
    return df

In [85]:
eco_df = load_combined_eco_demo(data_csvs[1:5])

In [86]:
eco_df.head()

,Geography,Total Male,Total Male Under 5 years,Total Male 5 to 9 years,Total Male 10 to 14 years,Total Male 15 to 17 years,Total Male 18 and 19 years,Total Male 20 years,Total Male 21 years,Total Male 22 to 24 years,...,"Total 12th grade, no diploma",Total Regular high school diploma,Total GED or alternative credential,"Total Some college, less than 1 year","Total Some college, 1 or more years, no degree",Total Associate's degree,Total Bachelor's degree,Total Master's degree,Total Professional school degree,Total Doctorate degree
0,0500000US01001,27052,1727,2156,1674,1208,553,395,225,1127,...,668,9663,2217,2194,5469,3323,6320,3399,504,498
1,0500000US01003,105889,6082,5197,8289,4305,2406,1177,792,3581,...,3403,34521,7751,10511,23964,14357,31444,13331,2845,2016
2,0500000US01005,13156,660,741,721,467,261,365,103,432,...,489,5171,1190,1018,2469,1385,1296,540,118,113
3,0500000US01007,12022,626,693,621,569,323,111,71,524,...,368,5879,1327,768,2056,1087,1183,481,109,41
4,0500000US01009,28677,1844,1738,2034,1257,743,378,312,1023,...,779,10582,3393,3415,5090,5220,3540,1382,222,132


## Demographic Data

This dataset requires same cleaning and loading as the economic dataset.

In [87]:
dem_df = load_combined_eco_demo(data_csvs[5:])

In [88]:
dem_df.head()

,Geography,Total Income in the past 12 months below poverty level,Total Income in the past 12 months below poverty level Male,Total Income in the past 12 months below poverty level Male Under 5 years,Total Income in the past 12 months below poverty level Male 5 years,Total Income in the past 12 months below poverty level Male 6 to 11 years,Total Income in the past 12 months below poverty level Male 12 to 14 years,Total Income in the past 12 months below poverty level Male 15 years,Total Income in the past 12 months below poverty level Male 16 and 17 years,Total Income in the past 12 months below poverty level Male 18 to 24 years,...,Total Income in the past 12 months at or above poverty level Female 55 to 64 years,Total Income in the past 12 months at or above poverty level Female 65 to 74 years,Total Income in the past 12 months at or above poverty level Female 75 years and over,Median household income in the past 12 months (in 2020 inflation-adjusted dollars),Total In labor force,Total In labor force Civilian labor force,Total In labor force Civilian labor force Employed,Total In labor force Civilian labor force Unemployed,Total In labor force Armed Forces,Total Not in labor force
0,0500000US01001,8394,3417,257,55,362,244,104,226,357,...,3005,2460,1809,57982,26025,25316,24580,736,709,18335
1,0500000US01003,19739,7803,563,176,462,420,166,267,971,...,14316,13017,8410,61756,103116,102795,98768,4027,321,73736
2,0500000US01005,6312,2509,440,94,272,199,143,37,440,...,1221,1279,766,34990,9356,9356,8707,649,0,11051
3,0500000US01007,3755,1865,188,43,234,86,136,111,225,...,1095,957,820,51721,8970,8970,8303,667,0,9451
4,0500000US01009,7847,3278,278,72,423,117,45,137,220,...,3543,2668,1914,48922,24133,24089,22836,1253,44,21950


In [89]:
int('010001')

10001

- my goals
- i do linear
- atif does lcassifcation

Make the below as the project goal

- split by year where 2024 is the prediction data set for all models
- 2020 and previous is the data set

once the data is merged and cleaned, people get data sets and do prediction on different years for their model
